In [20]:
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import shap
from xgboost import XGBClassifier
# import random forest regressor
from sklearn.ensemble import RandomForestRegressor
#import linear regression
from sklearn.linear_model import LinearRegression
# import tqdm
from tqdm import tqdm
import tqdm
#import r2_score
from sklearn.metrics import r2_score
#import confusion matrix
from sklearn.metrics import confusion_matrix
# import roc auc score
from sklearn.metrics import roc_auc_score
from food_crisis_functions import *
import json
with open("forecasting_hyperparameters.json", "r") as file:
    best_params_xgb_regressor= json.load(file)
    
with open("forecasting_hyperparameters_p3.json", "r") as file:
    best_params_xgb_regressor_for_p3= json.load(file)


In [21]:

# read csv
df = pd.read_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\forecasting_subset_IPCCH_v1210.csv')
# add dummys for area_id and month year
#df = pd.concat([df, pd.get_dummies(df['area_id'], prefix='area_id')], axis=1)
#df = pd.concat([df, pd.get_dummies(df['date'], prefix='month_year')], axis=1)
# drop lat and lon
#df = df.drop(['lat', 'lon'], axis=1)
###drop fews_ipc_ha
#df = df.drop(['fews_ipc_ha'], axis=1)
# random split train and test

# prepare date from year and month
df['date'] = pd.to_datetime(df[['year', 'month']].assign(DAY=1))
# check for inf and replace with na
df = df.replace([np.inf, -np.inf], np.nan)

# replace df['overall_phase], 0 as 1, >5 as 5
df['overall_phase'] = df['overall_phase'].apply(lambda x: 1 if x < 1 else (5 if x > 5 else x))

df_origin = df.copy()
y_pred_test = pd.DataFrame()
model_stats = pd.DataFrame()
#select phase1_percent is not na
df = df_origin[df_origin['phase1_percent'].notna()]

# Sort by region and date
df = df.sort_values(by=['area_id', 'date'])


In [22]:
# keep only date, overall phase and area_id
df_model = df[['date', 'overall_phase', 'area_id']]

In [23]:
# group by area_id, mark 1 if overall_phase >=4 at least once
df_model['phase4_experienced'] = df_model.groupby('area_id')['overall_phase'].transform(lambda x: x.ge(4).cummax())

C:\Users\swl00\AppData\Local\Temp\ipykernel_36668\454212809.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model['phase4_experienced'] = df_model.groupby('area_id')['overall_phase'].transform(lambda x: x.ge(4).cummax())


In [24]:
# read file
IPC_PATh = r'c:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\IPCCH_2017_2025_final_v12102025_with_zscores.csv'


df_IPC = pd.read_csv(IPC_PATh)

In [25]:
# for df_IPC, only keep area_id and country
df_IPC = df_IPC[['admin_code', 'country']]

In [26]:
# drop duplicates
df_IPC = df_IPC.drop_duplicates(subset=['admin_code'])

In [27]:
#rename admin_code to area_id
df_IPC = df_IPC.rename(columns={'admin_code': 'area_id'})

# merge df_model with df_IPC on area_id
df_model = df_model.merge(df_IPC, on='area_id', how='left')

In [13]:
# summarize by country, see phase_4_experienced>=1 or not
df_summary = df_model.groupby('country')['phase4_experienced'].max().reset_index()

In [15]:
df_summary['phase4_experienced'].mean()

0.4716981132075472

In [16]:
#extract overall_phase and country columns from df_model    
df_model_2 = df_model[['overall_phase', 'country']]

In [17]:
# group by country, mark 1 if overall_phase >=4 at least once
df_model_2['phase4_experienced'] = df_model_2.groupby('country')['overall_phase'].transform(lambda x: x.ge(4).cummax())

C:\Users\swl00\AppData\Local\Temp\ipykernel_36668\854496104.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model_2['phase4_experienced'] = df_model_2.groupby('country')['overall_phase'].transform(lambda x: x.ge(4).cummax())


In [18]:
# drop overall_phase
df_model_2 = df_model_2.drop(columns=['overall_phase'])

# group by country and get max
df_summary_2 = df_model_2.groupby('country')['phase4_experienced'].max().reset_index()

In [19]:
# mean
df_summary_2['phase4_experienced'].mean()

0.4716981132075472